In [ ]:
import time
import os

from pyspark.sql.functions import when, col, current_timestamp, to_json, struct
from pyspark.sql import SparkSession

from kafka import KafkaConsumer, KafkaProducer
from config import kafka_config
import json

In [ ]:
# Налаштування конфігурації SQL бази даних
jdbc_url = "jdbc:mysql://217.61.57.46:3306/olympic_dataset"
# jdbc_table = "athlete_bio"
jdbc_user = "neo_data_admin"
jdbc_password = "Proyahaxuqithab9oplp"

In [ ]:
# Створення Spark сесії
spark = SparkSession.builder \
    .config("spark.jars", "mysql-connector-j-8.0.32.jar") \
    .appName("JDBCToKafka") \
    .getOrCreate()

In [ ]:
# Створення Kafka Consumer
consumer = KafkaConsumer(
    bootstrap_servers=kafka_config['bootstrap_servers'],
    security_protocol=kafka_config['security_protocol'],
    sasl_mechanism=kafka_config['sasl_mechanism'],
    sasl_plain_username=kafka_config['username'],
    sasl_plain_password=kafka_config['password'],
    value_deserializer=lambda v: (json.loads(v.decode('utf-8')) if v != None else None),
    key_deserializer=lambda v: (json.loads(v.decode('utf-8')) if v != None else None),  
    auto_offset_reset='latest',  # Зчитування повідомлень з початку latest
    enable_auto_commit=True,       # Автоматичне підтвердження зчитаних повідомлень
    # group_id='my_consumer_group_3'   # Ідентифікатор групи споживачів
)

# контрольний ідентифікатор сесії передачі датасета через топік
CONTROL_SESSION_ID = 2

# Назва топіку
RESULT_TOPIC_NAME = 'athlete_event_results'

# Підписка на топік
consumer.subscribe([RESULT_TOPIC_NAME])
print(f"Subscribed to topic '{RESULT_TOPIC_NAME}'")

try:
    data_row_list = []
    message_numbers_list = []
    for message in consumer:
        data = message.value
        block_session_id = int(message.headers[0][1].decode('utf-8')) 
        full_data_len = int(message.headers[1][1].decode('utf-8'))
        block_start_position = int(message.headers[2][1].decode('utf-8'))
        block_finish_position = block_start_position + len(data)
        
        mask_set = set(range(full_data_len)) # маска повного пулу рядків таблиці
        message_numbers_list.extend(list(range(block_start_position, block_finish_position))) # список номерів отриманих рядків

        if block_session_id == CONTROL_SESSION_ID: # перевірка відповідності повідомлення в топіку до контрольного номера сесії
            print(
                f'session_id: {block_session_id},',
                f'block_start_position: {block_start_position},',
                f'block_finish_position: {block_finish_position-1}',
            )
            data_row_list.extend(message.value)
            
            if len(list(mask_set - set(message_numbers_list))) == 0: #якщо отримали повний набір даних - закриваємо консюмер
                consumer.close()
                break
            
except Exception as e:
    print(f"An error occurred: {e}")
    consumer.close()
    print("Consumer stopped its work.")

recieved_df = spark.createDataFrame(data_row_list)
print(f'Recieved {recieved_df.count()} rows of table')


**!!! Now start all cells of 3_sender.ipynb for sending messages from result mysql table**

In [ ]:
# Етап 1. Зчитати дані фізичних показників атлетів за допомогою Spark з MySQL таблиці olympic_dataset.athlete_bio

# Читання bio-даних з SQL бази даних
bio_df = spark.read.format('jdbc').options(
    url=jdbc_url,
    driver='com.mysql.cj.jdbc.Driver',  # com.mysql.jdbc.Driver
    dbtable="athlete_bio",
    user=jdbc_user,
    password=jdbc_password) \
    .load()

In [ ]:
# Етап 2. Відфільтрувати дані, де показники зросту та ваги є порожніми або не є числами.

# Заміна текстових значень на None перед перетворенням типу даних, фільтрація ненульових значень ваги та зросту
bio_df = bio_df.withColumn(
    "height",
    when(col("height").rlike("^[0-9]+(\\.[0-9]+)?$"), col("height").cast("float")).otherwise(None)
)
bio_df = bio_df.filter(col("height").isNotNull())

bio_df = bio_df.withColumn(
    "weight",
    when(col("weight").rlike("^[0-9]+(\\.[0-9]+)?$"), col("weight").cast("integer")).otherwise(None)
)
bio_df = bio_df.filter(col("weight").isNotNull())
print(f'bio_df cleaned')

In [ ]:
# Етап 4. Об’єднати дані з результатами змагань з Kafka-топіку з біологічними даними з MySQL таблиці за допомогою ключа athlete_id.

# Виконання left join
df_joined = recieved_df.join(bio_df.select("athlete_id", "height", "weight", 'sex'), on=['athlete_id'], how='left')
print(f'tables joined')

In [ ]:
# Етап 5. Знайти середній зріст і вагу атлетів індивідуально для кожного виду спорту, типу медалі або її відсутності, статі, країни (country_noc). 
# Додайте також timestamp, коли розрахунки були зроблені.

df_grouped = df_joined.groupBy("sport", 'medal', 'sex', 'country_noc').avg("height", 'weight') \
    .withColumn('timestamp', current_timestamp()) \
    .withColumnRenamed("avg(height)", "avg_height") \
    .withColumnRenamed("avg(weight)", "avg_weight")
df_grouped.show()

In [ ]:
 # Збереження в тимчасовий файл в початковій Spark-сесії для відкриття в іншій спарк-сесії, а також для читання як потокового датафрейму
df_grouped.write.mode("overwrite").parquet("temp.parquet")
print(f'temp.parquet saved')

In [ ]:
#  Пакет, необхідний для читання Kafka зі Spark
os.environ[
    'PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.1,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 pyspark-shell'

# Створення SparkSession
spark= (SparkSession.builder
         .appName("KafkaStreaming")
         .config("spark.streaming.backpressure.enabled", "true")
         .master("local[*]")
         .getOrCreate())

In [ ]:
# завантаження тимчасового файлу для завантаження схеми диних
new_session_df = spark.read.parquet("temp.parquet")
new_session_df.show(5)
print(new_session_df.schema)

In [ ]:
# стрімінг в консоль
# schema = new_session_df.schema
# streaming_df = spark.readStream.schema(schema).format("parquet").load("temp.parquet")

# _ = (streaming_df.writeStream
#   .outputMode("append")
#   .format("console") # Або "kafka", "parquet", тощо
#   # .option("path", "output/stream_path") # Якщо пишемо у файли
#   .trigger(once=True) # Або .trigger(availableNow=True) для Spark 3.3+
#   .start().awaitTermination())

In [49]:
# стрімінг в топік

# схема даних
schema = new_session_df.schema
# завантаження в потоковий датафрейм
streaming_df = spark.readStream.schema(schema).format("parquet").load("temp.parquet")

# streaming_df.count()

# Назва топіку
my_name = "vasyliev_v"
topic_name = f'{my_name}_alerting'

table_name = 'avg_stats_vvv'


# надсилання агрегованих та відфільтрованих даних до вихідного топіку


def process_batch(batch_df, epoch_id):

# 6. а) Зробіть стрим даних (за допомогою функції forEachBatch) вихідний кафка-топік,
    
    try:
        # Фільтрація пустих батчів, щоб не перевантажувати вихідний топік
        if not batch_df.isEmpty():  # Перевірка на порожній DataFrame
            # Перетворення даних у JSON для колонки 'value'
            kafka_df = batch_df.select(
                                        to_json(struct("*")).alias("value")
                                    )
            
            kafka_df.write.format("kafka") \
                .option("kafka.bootstrap.servers", "77.81.230.104:9092") \
                .option("kafka.security.protocol", "SASL_PLAINTEXT") \
                .option("kafka.sasl.mechanism", "PLAIN") \
                .option("kafka.sasl.jaas.config",
                        "org.apache.kafka.common.security.plain.PlainLoginModule required username='admin' password='VawEzo1ikLtrA8Ug8THa';") \
                .option("topic", topic_name) \
                .option("checkpointLocation", "/tmp/checkpoints-33") \
                .save()
    except Exception as e:
        print(f"Помилка під час запису даних у Kafka: {e}")

# 6. б) Зробіть стрим даних (за допомогою функції forEachBatch) у базу даних.
    
    #Збереження збагачених даних до MySQL
    try:
        batch_df.write \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("driver", 'com.mysql.cj.jdbc.Driver') \
            .option("dbtable", table_name) \
            .option("user", jdbc_user) \
            .option("password", jdbc_password) \
            .mode("append") \
            .save()
    except Exception as e:
        print(f"Помилка під час запису даних у MySQL: {e}")

# Запис оброблених даних у вихідний топік
query = streaming_df.writeStream \
    .foreachBatch(process_batch) \
    .trigger(availableNow=True) \
    .outputMode("append") \
    .start() \
    .awaitTermination()